In [ ]:
from dataclasses import dataclass


@dataclass
class LMConfigExample:
    vocab_size: int = 50257
    context_length: int = 1024
    num_layers: int = 48
    d_model: int = 1600
    d_k: int = 1600
    num_heads: int = 25
    d_ff: int = 4288


lm_config = LMConfigExample()

In [4]:
def calculate_trainable_params(config: LMConfigExample) -> int:

    rmsnorm_params = 1 * config.d_model
    MHSA_params = 4 * config.d_model * config.d_model
    SwiGLU_params = 3 * config.d_model * config.d_ff
    transformer_block_params = 2 * rmsnorm_params + MHSA_params + SwiGLU_params

    embed_params = config.vocab_size * config.d_model
    lmhead_params = config.vocab_size * config.d_model

    return (
        embed_params
        + config.num_layers * transformer_block_params
        + rmsnorm_params
        + lmhead_params
    )


trainable_params = calculate_trainable_params(lm_config)
float_32_bytes = 4
print(f"{trainable_params / 1e9} B")
print(f"{trainable_params * 4 / 1e9} GB")

1.6404528 B
6.5618112 GB


In [5]:
# def calculate_theoretical_forward_flops_linears(config: LMConfigExample) -> int:
#     return 2 * config.context_length * calculate_trainable_params(config)

# print(f"{calculate_theoretical_forward_flops_linears(lm_config) / 1e12} TeraFLOPs")

In [ ]:
def calculate_matmul_flops(config: LMConfigExample):
    qkvo_flops = 4 * config.context_length * config.d_model * config.d_k

    attention_flops = (
        2 * config.context_length * config.d_k * config.context_length  # qk v
    )
    swiglu_flops = 3 * (
        config.d_model * config.context_length * config.d_ff
    )  # W1 W2 W3
    lm_head_flops = config.context_length * config.d_model * config.vocab_size

    return (
        (qkvo_flops + attention_flops + swiglu_flops) * config.num_layers
        + lm_head_flops
    ) * 2  # multiply and add


print(f"{calculate_matmul_flops(lm_config) / 1e12} TeraFLOPs")

3.5167698944 TeraFLOPs
